# 02 — Fast Dataset Integrity Gate

This replaces the slow full-image verification pass. It preserves the V2 + L1 workflow while using the dataset's documented provenance, exact count checks, and a stratified readability sample.


In [4]:
# Notebook 02 — Fast Dataset Integrity Gate
# V2 + L1 workflow
#
# Purpose:
#   - Verify the locked directory structure and exact dataset counts.
#   - Verify the dataset-provided provenance files.
#   - Perform a stratified readability sample instead of decoding all 304,154 images.
#
# IMPORTANT:
#   This notebook does NOT modify, move, rename, hash, or re-save images.
#   Full pixel decoding still occurs later during feature extraction (Notebooks 04/05).
#
# Why a sample here?
#   Notebook 01 already performs the fast filesystem/count audit.
#   The dataset metadata already documents duplicate/invalid-image processing.
#   Opening and verifying every image here is redundant and extremely slow.
#   We therefore use a statistically useful stratified readability gate.

from pathlib import Path
import csv
import json
import os
import random
import time

from PIL import Image, ImageOps
from tqdm.auto import tqdm

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    # Fallback: locate project root from this notebook's working directory.
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data").exists():
            ROOT = p
            break

DATASET = ROOT / "data" / "raw" / "deepfake_merged_dataset"
METADATA = DATASET / "metadata"

EXPECTED = {
    "train": {"real": 102358, "fake": 128003},
    "val":   {"real": 19853,  "fake": 28436},
    "test":  {"real": 11199,  "fake": 14305},
}

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
SAMPLE_PER_CLASS_PER_SPLIT = 100
RANDOM_SEED = 42

print("PROJECT_ROOT:", ROOT)
print("DATASET:", DATASET)
print("SAMPLE_PER_CLASS_PER_SPLIT:", SAMPLE_PER_CLASS_PER_SPLIT)
print()

assert DATASET.exists(), f"Dataset not found: {DATASET}"

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------
def image_paths(folder):
    return sorted(
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

def count_images(folder):
    return sum(
        1 for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

def read_csv_rows(path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        return list(csv.reader(f))

def verify_one_image(path):
    """
    Lightweight but real readability check:
      - PIL open
      - EXIF transpose
      - load pixel data
      - basic geometry check
    No image is modified or saved.
    """
    try:
        with Image.open(path) as im:
            im = ImageOps.exif_transpose(im)
            im.load()
            if im.width <= 0 or im.height <= 0:
                return False, "invalid_dimensions"
            return True, ""
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"

# ---------------------------------------------------------------------
# 1. Directory structure gate
# ---------------------------------------------------------------------
required_dirs = [
    DATASET / "train" / "real",
    DATASET / "train" / "fake",
    DATASET / "val" / "real",
    DATASET / "val" / "fake",
    DATASET / "test" / "real",
    DATASET / "test" / "fake",
]

missing = [str(p) for p in required_dirs if not p.exists()]
assert not missing, "Missing required directories:\n" + "\n".join(missing)

print("1) DIRECTORY STRUCTURE: PASSED")

# ---------------------------------------------------------------------
# 2. Exact count gate
# ---------------------------------------------------------------------
print("\n2) EXACT DATASET COUNTS")
actual_counts = {}

for split, classes in EXPECTED.items():
    actual_counts[split] = {}
    for cls, expected_count in classes.items():
        folder = DATASET / split / cls
        actual = count_images(folder)
        actual_counts[split][cls] = actual
        status = "OK" if actual == expected_count else "MISMATCH"
        print(f"{split:5s} / {cls:4s}: {actual:7d}  expected={expected_count:7d}  [{status}]")
        assert actual == expected_count, (
            f"Count mismatch for {split}/{cls}: "
            f"actual={actual}, expected={expected_count}"
        )

total_actual = sum(v for split in actual_counts.values() for v in split.values())
total_expected = sum(v for split in EXPECTED.values() for v in split.values())

print(f"\nTOTAL: {total_actual:,}  expected={total_expected:,}")
assert total_actual == total_expected == 304154

# ---------------------------------------------------------------------
# 3. Dataset metadata/provenance gate
# ---------------------------------------------------------------------
print("\n3) DATASET PROVENANCE FILES")

required_metadata = [
    METADATA / "manifest.csv",
    METADATA / "dataset_summary.json",
    METADATA / "class_map.json",
    METADATA / "sampling_policy.json",
    METADATA / "duplicates_skipped.csv",
    METADATA / "invalid_images.csv",
]

for p in required_metadata:
    exists = p.exists()
    print(f"{p.name:24s}: {'FOUND' if exists else 'MISSING'}")
    assert exists, f"Required provenance file missing: {p}"

# Confirm invalid_images.csv does not document invalid images in the final build.
invalid_path = METADATA / "invalid_images.csv"
invalid_rows = read_csv_rows(invalid_path)

# Allow a header-only CSV. If there are data rows, they represent documented invalid files.
invalid_data_rows = invalid_rows[1:] if invalid_rows else []
print(f"\ninvalid_images.csv data rows: {len(invalid_data_rows)}")

assert len(invalid_data_rows) == 0, (
    "Dataset provenance reports invalid images. "
    "Inspect metadata/invalid_images.csv before continuing."
)

# ---------------------------------------------------------------------
# 4. Class map sanity check
# ---------------------------------------------------------------------
class_map_path = METADATA / "class_map.json"
with class_map_path.open("r", encoding="utf-8") as f:
    class_map = json.load(f)

print("\n4) CLASS MAP")
print(json.dumps(class_map, indent=2))

# We require the project convention used by the ML notebooks:
# real=0, fake=1.
def normalize_class_map(obj):
    if isinstance(obj, dict):
        flat = {str(k).lower(): str(v).lower() for k, v in obj.items()}
        return flat
    return {}

flat_map = normalize_class_map(class_map)

# Do not fail merely because the JSON uses a different nesting/layout.
# The actual folder labels are authoritative for this project.
print("Folder-label convention used downstream: real=0, fake=1")

# ---------------------------------------------------------------------
# 5. Stratified readability sample
# ---------------------------------------------------------------------
print("\n5) STRATIFIED IMAGE READABILITY SAMPLE")

rng = random.Random(RANDOM_SEED)
sample_records = []
failures = []

start = time.perf_counter()

for split, classes in EXPECTED.items():
    for cls in classes:
        folder = DATASET / split / cls
        paths = image_paths(folder)

        assert len(paths) == EXPECTED[split][cls]

        n = min(SAMPLE_PER_CLASS_PER_SPLIT, len(paths))
        selected = rng.sample(paths, n)

        split_cls_ok = 0
        for p in tqdm(
            selected,
            desc=f"{split}/{cls}",
            unit="img",
            leave=False,
        ):
            ok, err = verify_one_image(p)
            sample_records.append({
                "split": split,
                "class": cls,
                "path": str(p),
                "ok": int(ok),
                "error": err,
            })
            if ok:
                split_cls_ok += 1
            else:
                failures.append((p, err))

        print(f"{split}/{cls}: {split_cls_ok}/{n} readable")

elapsed = time.perf_counter() - start

assert not failures, (
    "Readability sample found failures:\n" +
    "\n".join(f"{p} -> {err}" for p, err in failures[:20])
)

sample_total = len(sample_records)
print(f"\nSample checked: {sample_total} images")
print(f"Elapsed: {elapsed:.1f} seconds")
print("Readability sample: PASSED")

# ---------------------------------------------------------------------
# 6. Write audit result
# ---------------------------------------------------------------------
metrics_dir = ROOT / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)

out_csv = metrics_dir / "fast_integrity_sample_v2.csv"
with out_csv.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["split", "class", "path", "ok", "error"]
    )
    writer.writeheader()
    writer.writerows(sample_records)

summary = {
    "dataset_total": total_actual,
    "expected_total": total_expected,
    "counts_match": True,
    "invalid_provenance_rows": len(invalid_data_rows),
    "sample_per_class_per_split": SAMPLE_PER_CLASS_PER_SPLIT,
    "sample_total": sample_total,
    "sample_failures": len(failures),
    "random_seed": RANDOM_SEED,
    "status": "PASSED",
}

summary_path = metrics_dir / "fast_integrity_gate_v2.json"
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\n============================================================")
print("FAST DATASET INTEGRITY GATE PASSED")
print("============================================================")
print(f"Total images: {total_actual:,}")
print(f"Readability sample: {sample_total:,}")
print(f"Sample failures: {len(failures)}")
print(f"Audit CSV: {out_csv}")
print(f"Audit JSON: {summary_path}")
print()
print("Proceed to Notebook 03.")


PROJECT_ROOT: d:\deepfake_noise_wavelet_ml
DATASET: d:\deepfake_noise_wavelet_ml\data\raw\deepfake_merged_dataset
SAMPLE_PER_CLASS_PER_SPLIT: 100

1) DIRECTORY STRUCTURE: PASSED

2) EXACT DATASET COUNTS
train / real:  102358  expected= 102358  [OK]
train / fake:  128003  expected= 128003  [OK]
val   / real:   19853  expected=  19853  [OK]
val   / fake:   28436  expected=  28436  [OK]
test  / real:   11199  expected=  11199  [OK]
test  / fake:   14305  expected=  14305  [OK]

TOTAL: 304,154  expected=304,154

3) DATASET PROVENANCE FILES
manifest.csv            : FOUND
dataset_summary.json    : FOUND
class_map.json          : FOUND
sampling_policy.json    : FOUND
duplicates_skipped.csv  : FOUND
invalid_images.csv      : FOUND

invalid_images.csv data rows: 0

4) CLASS MAP
{
  "real": 0,
  "fake": 1
}
Folder-label convention used downstream: real=0, fake=1

5) STRATIFIED IMAGE READABILITY SAMPLE


train/real:   0%|          | 0/100 [00:00<?, ?img/s]

train/real: 100/100 readable


train/fake:   0%|          | 0/100 [00:00<?, ?img/s]

train/fake: 100/100 readable


...

val/fake: 100/100 readable


test/real:   0%|          | 0/100 [00:00<?, ?img/s]

test/real: 100/100 readable


test/fake:   0%|          | 0/100 [00:00<?, ?img/s]

test/fake: 100/100 readable

Sample checked: 600 images
Elapsed: 21.1 seconds
Readability sample: PASSED

FAST DATASET INTEGRITY GATE PASSED
Total images: 304,154
Readability sample: 600
Sample failures: 0
Audit CSV: d:\deepfake_noise_wavelet_ml\metrics\fast_integrity_sample_v2.csv
Audit JSON: d:\deepfake_noise_wavelet_ml\metrics\fast_integrity_gate_v2.json

Proceed to Notebook 03.
